In [ ]:
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
!pip install gradio -q
!pip install demucs -q
!pip install stable-ts -q
!pip install openai -q
!pip install pydub -q
!pip install coqui-tts -q

# 2. IMPORTS E CONFIGURAÇÃO
import gradio as gr
import os
import shutil
import subprocess
import stable_whisper
from openai import OpenAI
import torch
from pydub import AudioSegment
import math
import traceback

# !! IMPORTANTE: Resolve a necessidade de digitar 'y' para os termos da Coqui TTS !!
# Esta linha DEVE vir ANTES de importar a biblioteca TTS.
os.environ['COQUI_TOS_AGREED'] = '1'

from TTS.api import TTS

print("✅ Ambiente final pronto com todas as dependências!")

In [ ]:
# 3. CARREGAMENTO DOS MODELOS DE IA (WHISPER E TTS)
print("Carregando modelos de IA... (Isso pode levar vários minutos)")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Carrega Whisper
print("-> Carregando Stable Whisper...")
whisper_model = stable_whisper.load_model('medium')
print("✅ Modelo Whisper carregado.")

# Carrega Coqui TTS
print("-> Carregando Coqui XTTS...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Modelo Coqui TTS carregado.")

In [ ]:
# 4. FUNÇÕES AUXILIARES

def speed_change(sound, speed=1.0):
    """Altera a velocidade de um segmento de áudio Pydub via reamostragem."""
    sound_with_altered_frame_rate = sound._spawn(sound.raw_data, overrides={"frame_rate": int(sound.frame_rate * speed)})
    return sound_with_altered_frame_rate.set_frame_rate(sound.frame_rate)

def criar_status_html(icon, text, css_class=""):
    """Cria o bloco HTML para um passo do status."""
    return f'<div class="status-step {css_class}"><div class="icon">{icon}</div><div class="text">{text}</div></div>'

print("✅ Funções auxiliares definidas.")

In [37]:
class MusicTranslatorApp:
    CSS = """.status-container { display: flex; justify-content: space-around; align-items: center; width: 100%; gap: 10px; flex-wrap: wrap; } .status-step { text-align: center; font-weight: bold; color: #888; padding: 10px; border: 3px solid #DDD; border-radius: 15px; flex: 1; min-width: 100px; transition: all 0.3s ease-in-out; } .status-step.active { border-color: #3B82F6; color: #3B82F6; transform: scale(1.05); box-shadow: 0 0 15px rgba(59, 130, 246, 0.4); } .status-step.completed { border-color: #16A34A; color: #16A34A; } .status-step.error { border-color: #EF4444; color: #EF4444; } .status-step .icon { font-size: 2em; } .status-step .text { font-size: 0.9em; }"""

    def __init__(self):
        self.demo = self._build_ui()

    def _main_pipeline(self, audio_path, idioma_original, idioma_alvo, openai_api_key, ajuste_volume_db):
        """Pipeline principal que 'yielda' atualizações para a interface."""

        # Mapa dos componentes de status para fácil acesso
        status_map = {
            'separation': {'elem': self.status_separation, 'icon': '🎵✂️', 'text': 'Separação'},
            'translation': {'elem': self.status_translation, 'icon': '🌐', 'text': 'Tradução'},
            'assembly': {'elem': self.status_assembly, 'icon': '🎤', 'text': 'Geração Vocal'},
            'mixing': {'elem': self.status_mixing, 'icon': '🎚️', 'text': 'Mixagem Final'},
            'result': {'elem': self.status_result, 'icon': '🎶', 'text': 'Resultado'}
        }
        active_step_key = None

        try:
            # ETAPA 1: SEPARAÇÃO
            active_step_key = 'separation'
            yield {**self._reset_ui(), status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "active")), self.status_main: gr.Textbox(value="[1/5] Iniciando separação de faixas com Demucs...")}
            output_dir, base_name = "/content/separated", os.path.splitext(os.path.basename(audio_path))[0]
            if os.path.exists(output_dir): shutil.rmtree(output_dir)
            command = f'python3 -m demucs --two-stems vocals -o "{output_dir}" "{audio_path}"'
            subprocess.run(command, shell=True, check=True, text=True, capture_output=True)
            result_path = os.path.join(output_dir, "htdemucs_ft", base_name)
            if not os.path.exists(result_path): result_path = os.path.join(output_dir, "htdemucs", base_name)
            caminho_vocal, caminho_instrumental = os.path.join(result_path, "vocals.wav"), os.path.join(result_path, "no_vocals.wav")

            # ETAPA 2: TRADUÇÃO
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "completed"))}
            active_step_key = 'translation'
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "active")), self.status_main: gr.Textbox(value="[2/5] Transcrevendo e traduzindo a letra...")}
            # ... (Lógica de Whisper e OpenAI) ...
            trans_result = whisper_model.transcribe(caminho_vocal, language=idioma_original, regroup=True); segments_original = trans_result.segments
            if not openai_api_key:
              raise gr.Error("Chave da API da OpenAI não fornecida.");

            client = OpenAI(api_key=openai_api_key)

            lyrics_numbered = "".join([f"{i}: {s.text.strip()}\n" for i, s in enumerate(segments_original)])
            prompt = f"Você é um tradutor especialista em letras de música, traduzindo do idioma '{idioma_original}' para '{idioma_alvo}'. Mantenha a poesia, ritmo e significado. Regras: adapte, não traduza literalmente; tente manter a métrica; refrões consistentes; retorne a tradução mantendo EXATAMENTE a mesma numeração (formato 'NÚMERO: Tradução')."
            response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "system", "content": prompt}, {"role": "user", "content": lyrics_numbered}], temperature=0.3)
            raw_translation = response.choices[0].message.content
            mapped_translations = {int(p[0]): p[1].strip() for ln in raw_translation.strip().split('\n') if len(p := ln.split(':', 1)) == 2}
            segments_translated = [{"inicio": s.start, "fim": s.end, "texto_traduzido": mapped_translations.get(i, "")} for i, s in enumerate(segments_original)]


            # ETAPA 3: GERAÇÃO VOCAL
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "completed"))}
            active_step_key = 'assembly'
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "active")), self.status_main: gr.Textbox(value="[3/5] Gerando e alinhando nova voz (TTS)...")}
            # ... (Lógica de TTS e alinhamento) ...
            total_duration_ms = math.ceil(segments_translated[-1]['fim']) * 1000; final_vocal_track = AudioSegment.silent(duration=total_duration_ms); temp_tts_dir = "segmentos_tts"; os.makedirs(temp_tts_dir, exist_ok=True)
            for seg in segments_translated:
                original_duration_s = seg['fim'] - seg['inicio']
                if original_duration_s < 0.2 or not seg['texto_traduzido']: continue
                temp_path = os.path.join(temp_tts_dir, "temp_seg.wav"); tts_model.tts_to_file(text=seg['texto_traduzido'], speaker_wav=caminho_vocal, language=idioma_alvo, file_path=temp_path)
                generated_audio = AudioSegment.from_wav(temp_path);
                if len(generated_audio) == 0: continue
                generated_duration_s = len(generated_audio) / 1000.0; speed_factor = generated_duration_s / original_duration_s if original_duration_s > 0 else 1
                if speed_factor == 0: continue
                aligned_audio = generated_audio.speedup(playback_speed=speed_factor) if speed_factor > 1.0 else speed_change(generated_audio, speed_factor)
                final_vocal_track = final_vocal_track.overlay(aligned_audio, position=seg['inicio'] * 1000)
            caminho_vocal_gerado = "/content/vocal_traduzido_final.wav"; final_vocal_track.export(caminho_vocal_gerado, format="wav")


            # ETAPA 4: MIXAGEM FINAL
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "completed"))}
            active_step_key = 'mixing'
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "active")), self.status_main: gr.Textbox(value="[4/5] Mixando faixas...")}
            # ... (Lógica de Mixagem) ...
            instrumental = AudioSegment.from_wav(caminho_instrumental); vocal_ajustado = AudioSegment.from_wav(caminho_vocal_gerado) + ajuste_volume_db
            final_music = instrumental.overlay(vocal_ajustado); caminho_musica_final = "musica_final_completa.mp3"; final_music.export(caminho_musica_final, format="mp3", bitrate="192k")

            # ETAPA 5: SUCESSO
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "completed"))}
            active_step_key = 'result'
            yield {status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "active")), self.status_main: gr.Textbox(value="[5/5] Processo concluído com sucesso!"), self.output_final_song: gr.Audio(value=caminho_musica_final, visible=True)}

        except Exception as e:
            error_message = f"Erro na etapa '{status_map[active_step_key]['text']}': {e}"
            traceback.print_exc()
            yield {self.status_main: gr.Textbox(value=error_message), status_map[active_step_key]['elem']: gr.HTML(criar_status_html(status_map[active_step_key]['icon'], status_map[active_step_key]['text'], "error"))}


    def _reset_ui(self):
        """Retorna o dicionário de componentes para o estado inicial."""
        return {
            self.status_separation: gr.HTML(criar_status_html("🎵✂️", "Separação")),
            self.status_translation: gr.HTML(criar_status_html("🌐", "Tradução")),
            self.status_assembly: gr.HTML(criar_status_html("🎤", "Geração Vocal")),
            self.status_mixing: gr.HTML(criar_status_html("🎚️", "Mixagem Final")),
            self.status_result: gr.HTML(criar_status_html("🎶", "Resultado")),
            self.output_final_song: gr.Audio(value=None, visible=False),
            self.status_main: gr.Textbox(value="Aguardando arquivo de áudio...", interactive=False)
        }

    def _build_ui(self):
        """Constrói a interface gráfica do Gradio."""
        with gr.Blocks(css=self.CSS, theme=gr.themes.Soft(primary_hue="blue", secondary_hue="sky")) as demo:
            gr.Markdown("# 🤖 Estúdio de Adaptação Musical com IA")
            gr.Markdown("#### Uma pipeline completa para traduzir e recriar músicas. Siga os passos e clique em 'Iniciar'.")

            with gr.Row(elem_classes="status-container"):
                self.status_separation = gr.HTML(criar_status_html("🎵✂️", "Separação"))
                self.status_translation = gr.HTML(criar_status_html("🌐", "Tradução"))
                self.status_assembly = gr.HTML(criar_status_html("🎤", "Geração Vocal"))
                self.status_mixing = gr.HTML(criar_status_html("🎚️", "Mixagem Final"))
                self.status_result = gr.HTML(criar_status_html("🎶", "Resultado"))

            self.status_main = gr.Textbox(label="Status do Processo", value="Aguardando arquivo...", interactive=False)

            with gr.Blocks():
                with gr.Row():
                    self.audio_input = gr.Audio(type="filepath", label="1. Envie sua Música")
                with gr.Row():
                    self.lang_original = gr.Dropdown(['en', 'pt', 'es', 'fr', 'de', 'ja', 'ru', 'it', 'ko', 'zh'], value='en', label="2. Idioma Original")
                    self.lang_target = gr.Dropdown(['pt', 'en', 'es', 'fr', 'de', 'ja', 'ru', 'it', 'ko', 'zh'], value='pt', label="3. Idioma da Tradução")
                with gr.Row():
                    self.openai_key = gr.Textbox(label="4. Chave da API OpenAI", type="password", placeholder="Cole sua chave sk-...")
                    self.volume_slider = gr.Slider(-12, 12, value=0, step=1, label="5. Ajuste de Volume do Vocal (dB)")
                with gr.Row():
                    self.process_button = gr.Button("▶️ Iniciar Processo Completo", variant="primary", scale=2)

            self.output_final_song = gr.Audio(label="Resultado Final", visible=False, interactive=False)

            all_outputs = [self.status_separation, self.status_translation, self.status_assembly, self.status_mixing, self.status_result, self.status_main, self.output_final_song]

            self.process_button.click(fn=self._main_pipeline, inputs=[self.audio_input, self.lang_original, self.lang_target, self.openai_key, self.volume_slider], outputs=all_outputs)
            self.audio_input.upload(fn=self._reset_ui, outputs=all_outputs)
        return demo

In [ ]:
if 'app' in locals():
    app.demo.close() # Fecha a instância anterior se estiver rodando para evitar conflitos de porta

app = MusicTranslatorApp()
app.demo.launch(share=True, debug=True)